In [0]:
%sql
/*create catalog sales_project;
create schema sales_project.brz;
create schema sales_project.slv;
create schema sales_project.gld;*/

In [0]:
# %sql
# truncate table sales_project.brz.bad_records_expenses;
# truncate table sales_project.brz.bad_records_sales;
# truncate table sales_project.brz.employees;
# truncate table sales_project.brz.expenses;
# truncate table sales_project.brz.regions;
# truncate table sales_project.brz.sales;
# truncate table sales_project.brz.state_store;
# truncate table sales_project.slv.employees_cdc;
# truncate table sales_project.slv.expenses;
# truncate table sales_project.slv.pipeline_metadata;
# truncate table sales_project.slv.sales;

In [0]:
ngrokip = "0.tcp.in.ngrok.io:15212"

checkpoints = {"sales": "/Workspace/Shared/checkpoints_sales_proj/sales_new",
                "employees": "/Workspace/Shared/checkpoints_sales_proj/emp_new",
                "regions": "/Workspace/Shared/checkpoints_sales_proj/regions_new",
                "expenses": "/Workspace/Shared/checkpoints_sales_proj/exp_new"}

topics = {"sales": "sales_new", "regions": "regions_new", "expenses": "expenses_new", "employees": "employees_new"}

In [0]:
%sql
/*create or replace table sales_project.brz.sales(
  key string,
  value string,
  topic string,
  partition int,
  offset bigint,
  timestamp timestamp
);
create or replace table sales_project.brz.expenses(
  key string,
  value string,
  topic string,
  partition int,
  offset bigint,
  timestamp timestamp
);
create or replace table sales_project.brz.regions(
  key string,
  value string,
  topic string,
  partition int,
  offset bigint,
  timestamp timestamp
);
create or replace table sales_project.brz.employees(
  key string,
  value string,
  topic string,
  partition int,
  offset bigint,
  timestamp timestamp
);*/

In [0]:
# dbutils.fs.rm(checkpoints["sales"], True)

In [0]:
from pyspark.sql import functions as F

def consume_data(topic_key, topic_name):

    df = (spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", ngrokip)
        .option("subscribe", topic_name)
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .load())

    df = df.selectExpr("CAST(key AS STRING) as key",
        "CAST(value AS STRING) as value",
        "topic",
        "partition",
        "offset",
        "timestamp").withColumn("ingestion_time", F.current_timestamp())

    query = (df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoints[topic_key])
        .trigger(availableNow=True)
        .outputMode("append")
        .table(f"sales_project.brz.{topic_key}"))
    
    return query

queries = {}

for topic_key, topic_name in topics.items():
    q = consume_data(topic_key, topic_name) #sales sales_new
    queries[topic_key] = q

for topic_key, q in queries.items():
    q.awaitTermination()
    print(f"Completed stream for: {topic_key}")

In [0]:
%sql
select * from sales_project.brz.sales;

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

slv_sales_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("sales_id", LongType(), False),
    StructField("employee_id", IntegerType(), False),
    StructField("region_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sales_amount", IntegerType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

# add ingestion filter here

df_raw = spark.read.table("sales_project.brz.sales")

df_raw = df_raw.select("value")

parsed_df = df_raw.withColumn("value", F.from_json(F.col("value"), slv_sales_schema))

parsed_df = parsed_df.select(F.col("value.sales_id").alias("sale_id"), F.col("value.employee_id").alias("employee_id"),
                       F.col("value.region_id").alias("region_id"), F.col("value.product_id"),
                       F.col("value.quantity").alias("quantity"),F.col("value.sales_amount").alias("amount"),
                       F.col("value.event_time").alias("event_time"), F.col("value.ingestion_time").alias("ingestion_time"))

last_processed = spark.sql("""select last_ingestion_time from sales_project.slv.pipeline_metadata
                           where topic_name = "sales";""").collect()[0]["last_ingestion_time"]

parsed_df = parsed_df.filter(F.col("ingestion_time")>F.to_timestamp(F.lit(last_processed)))

clean_df = (parsed_df.withWatermark("event_time", "2 hours").dropDuplicates(subset = ("sale_id",))
            .filter((F.col("sale_id").isNotNull()) & (F.col("employee_id").isNotNull()) & (F.col("region_id").isNotNull()) 
                    & (F.col("product_id").isNotNull()) & (F.col("quantity")>0) & (F.col("amount")>0))
            .withColumn("ingestion_time", F.current_timestamp()))
            

clean_df.write.format("delta").mode("append").saveAsTable("sales_project.slv.sales")

quarantine_df = (parsed_df
            .filter((F.col("sale_id").isNull()) | (F.col("employee_id").isNull()) | (F.col("region_id").isNull()) 
                    | (F.col("product_id").isNull()) | (F.col("quantity")<0) | (F.col("amount")<0))
            .drop(F.col("ingestion_time")))

quarantine_df.write.format("delta").mode("append").saveAsTable("sales_project.brz.bad_records_sales")

display(parsed_df)

spark.sql("""select * from sales_project.slv.sales;""")

In [0]:
batch_ingestion_time = parsed_df.agg(F.max(F.col("ingestion_time")).alias("ingestion_time"))\
    .withColumn("topic_name", F.lit("sales"))

batch_ingestion_time.createOrReplaceTempView("batch_ingestion_ts")

spark.sql("""merge into sales_project.slv.pipeline_metadata t
          using batch_ingestion_ts s
          on s.topic_name = t.topic_name
          when matched and s.ingestion_time is not null and
          (t.last_ingestion_time is null or s.ingestion_time>t.last_ingestion_time) then update
          set t.last_ingestion_time = s.ingestion_time
          when not matched then insert(last_ingestion_time, topic_name)
          values(s.ingestion_time, s.topic_name);""")

display(spark.read.table("sales_project.slv.pipeline_metadata"))
parsed_df.select("ingestion_time").show(5)

In [0]:
%sql
select * from sales_project.slv.sales
order by sale_id desc;

In [0]:
%sql
select count(*) from sales_project.slv.sales;


In [0]:
%sql
select * from sales_project.brz.employees
order by ingestion_time desc;

In [0]:
slv_employees_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("employee_id", IntegerType(), False),
    StructField("before", StructType([
        StructField("employee_name", StringType(), True),
        StructField("role", StringType(), True),
        StructField("department", StringType(), True),
        StructField("region_id", IntegerType(), True),
        StructField("joining_date", DateType(), True),
        StructField("salary", LongType(), True)
    ]), True),
    StructField("after", StructType([
        StructField("employee_name", StringType(), True),
        StructField("role", StringType(), True),
        StructField("department", StringType(), True),
        StructField("region_id", IntegerType(), True),
        StructField("joining_date", DateType(), True),
        StructField("salary", LongType(), True)
    ]), True),
    StructField("operation", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

emp_raw = spark.read.table("sales_project.brz.employees").select("value")

parsed_emp = emp_raw.withColumn("value", F.from_json(F.col("value"), slv_employees_schema))

parsed_emp = parsed_emp.select("value.employee_id", F.col("value.before.employee_name").alias("before_employee_name"),
                               F.col("value.before.role").alias("before_role"),
                               F.col("value.before.department").alias("before_department"),
                               F.col("value.before.region_id").alias("before_region_id"),
                               F.col("value.before.joining_date").alias("before_joining_date"),
                               F.col("value.before.salary").alias("before_salary"),
                               F.col("value.after.employee_name").alias("after_employee_name"),
                               F.col("value.after.role").alias("after_role"),
                               F.col("value.after.department").alias("after_department"),
                               F.col("value.after.region_id").alias("after_region_id"),
                               F.col("value.after.joining_date").alias("after_joining_date"),
                               F.col("value.after.salary").alias("after_salary"),
                               "value.operation", "value.event_time", "value.ingestion_time")

last_ingestion_time_emp = spark.sql("""select last_ingestion_time
                                    from sales_project.slv.pipeline_metadata
                                   where topic_name = "employees";""").collect()[0]["last_ingestion_time"]

parsed_emp = parsed_emp.filter(F.col("ingestion_time")>F.to_timestamp(F.lit(last_ingestion_time_emp)))


display(parsed_emp)

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy(F.col("employee_id"), F.col("operation")).orderBy(F.col("event_time").desc())

parsed_emp = parsed_emp.withColumn("rn", F.row_number().over(w)).filter(F.col("rn")==1).drop("rn")

inserted_emp = parsed_emp.filter(F.col("operation") == "I")\
                        .select("employee_id", F.initcap(F.col("after_employee_name")).alias("employee_name"), 
                                F.initcap(F.col("after_role")).alias("role"), F.initcap(F.col("after_department")).alias("department"),
                                F.col("after_region_id").alias("region_id"),
                                F.col("after_joining_date").alias("joining_date"),
                                F.col("after_salary").alias("salary"),
                                F.col("event_time").cast("date").alias("from_date"),
                                F.lit(None).cast("date").alias("to_date"), F.lit(True).alias("is_current"),
                                F.lit(False).alias("is_deleted"),
                                F.current_timestamp().alias("ingestion_time"))


inserted_emp.write.format("delta").mode("append").saveAsTable("sales_project.slv.employees_cdc")

## updated

updated_emp = parsed_emp.filter(F.col("operation") == "U")\
            .select("employee_id", F.initcap(F.col("after_employee_name")).alias("employee_name"),
                                F.initcap(F.col("after_role")).alias("role"),
                                F.initcap(F.col("after_department")).alias("department"),
                                F.col("after_region_id").alias("region_id"),
                                F.col("after_joining_date").alias("joining_date"),
                                F.col("after_salary").alias("salary"),
                                F.col("event_time").cast("date").alias("from_date"),
                                F.lit(None).cast("date").alias("to_date"), F.lit(True).alias("is_current"),
                                F.lit(False).alias("is_deleted"),
                                F.current_timestamp().alias("ingestion_time"))

updated_emp.createOrReplaceTempView("updated_empl")

# Step 1: Close old records (set to_date = new from_date - 1 day, is_current = False)
spark.sql("""
    MERGE INTO sales_project.slv.employees_cdc t
    USING updated_empl s
    ON s.employee_id = t.employee_id AND t.is_current = True
    WHEN MATCHED THEN
        UPDATE SET t.to_date = s.from_date, t.is_current = False
""")

# Step 2: Insert new versions with updated data
spark.sql("""
    INSERT INTO sales_project.slv.employees_cdc 
    (employee_id, employee_name, role, department, region_id, joining_date, salary, 
     from_date, to_date, is_current, ingestion_time)
    SELECT employee_id, employee_name, role, department, region_id, joining_date, salary,
           from_date, to_date, is_current, ingestion_time
    FROM updated_empl
""")

## deleted

# FIX: Use before_ columns for DELETE operations since after_ columns are NULL
deleted_emp = parsed_emp.filter(F.col("operation") == "D")\
            .select("employee_id", F.initcap(F.col("before_employee_name")).alias("employee_name"), 
                                F.initcap(F.col("before_role")).alias("role"),
                                F.initcap(F.col("before_department")).alias("department"),
                                F.col("before_region_id").alias("region_id"),
                                F.col("before_joining_date").alias("joining_date"),
                                F.col("before_salary").alias("salary"),
                                F.col("event_time").cast("date").alias("from_date"),
                                F.col("event_time").cast("date").alias("to_date"), F.lit(False).alias("is_current"),
                                F.current_timestamp().alias("ingestion_time"))
            
deleted_emp.createOrReplaceTempView("deleted_view")

spark.sql("""
    MERGE INTO sales_project.slv.employees_cdc t
    USING deleted_view s
    ON s.employee_id = t.employee_id AND t.is_current = True
    WHEN MATCHED THEN
        UPDATE SET t.to_date = s.to_date, t.is_current = False, t.is_deleted = True
""")

In [0]:
# from pyspark.sql.window import Window

# this code was just for initial processing, as haven't touched employeestable yet, but a lot of new records were created and duplicate inserts created too. Will process from now again, fixed upstream logic.

# parsed_emp = parsed_emp.filter(F.col("operation") == "I")\
#                         .select("employee_id", F.initcap(F.col("after_employee_name")).alias("employee_name"), 
#                                F.initcap(F.col("after_role")).alias("role"), F.initcap(F.col("after_department")).alias("department"),
#                                F.col("after_region_id").alias("region_id"),
#                                F.col("after_joining_date").alias("joining_date"),
#                                F.col("event_time").cast("date").alias("from_date"),
#                                F.lit(None).cast("date").alias("to_date"), F.lit(True).alias("is_current"),
#                                F.current_timestamp().alias("ingestion_time"))

# w = Window.partitionBy(F.col("employee_id")).orderBy(F.col("joining_date").asc())
# earliest = parsed_emp.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).drop("rn")

# earliest.write.format("delta").mode("append").saveAsTable("sales_project.slv.employees_cdc")

In [0]:
%sql
select * from sales_project.slv.employees_cdc
order by employee_id, is_current;

In [0]:
%sql

select count(*) from sales_project.slv.employees_cdc;

In [0]:
cdc_metadata = spark.read.table("sales_project.slv.employees_cdc")\
                .agg(F.max(F.col("ingestion_time")).alias("last_ingestion_time")).withColumn("topic", F.lit("employees"))

cdc_metadata.createOrReplaceTempView("cdc_metadata")

spark.sql("""merge into sales_project.slv.pipeline_metadata t
          using cdc_metadata s
          on s.topic=t.topic_name
          when matched and t.last_ingestion_time is not null and
          (s.last_ingestion_time>t.last_ingestion_time) then
          update set t.last_ingestion_time = s.last_ingestion_time, t.topic_name = s.topic
          when not matched then insert(topic_name, last_ingestion_time)
          values(s.topic, s.last_ingestion_time);""")

display(spark.sql("""select * from sales_project.slv.pipeline_metadata;"""))

In [0]:
%sql
-- create or replace table sales_project.slv.employees_cdc(
--     emp_sk BIGINT GENERATED ALWAYS AS IDENTITY,
--     employee_id int,
--     employee_name string,
--     role string,
--     department string,
--     region_id int,
--     joining_date date,
--     from_date date,
--     to_date date,
--     is_current boolean,
--     ingestion_time timestamp
-- )
-- using delta;

In [0]:
%sql
select * from sales_project.brz.expenses;

In [0]:
slv_exp_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("expense_id", IntegerType(), True),
    StructField("employee_id", StringType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("expense_type", StringType(), True),
    StructField("expense_amount", LongType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

exp_raw = spark.read.table("sales_project.brz.expenses").select("value")

exp_parsed = exp_raw.withColumn("value", F.from_json(F.col("value"), slv_exp_schema))

exp_parsed = exp_parsed.select("value.expense_id", "value.employee_id", "value.region_id", "value.expense_type",
                               "value.expense_amount", "value.event_time", "value.ingestion_time")
exp_parsed.display()

In [0]:
from pyspark.sql import functions as F

last_processed_expenseTs = spark.sql("""select last_ingestion_time
                                  from sales_project.slv.pipeline_metadata
                                  where topic_name = "expenses"; """).collect()[0]["last_ingestion_time"]

exp_parsed = exp_parsed.filter(F.col("ingestion_time")>F.to_timestamp(F.lit(last_processed_expenseTs)))

exp_parsed.display()

good_exp = exp_parsed.filter(F.col("expense_amount")>0)\
    .withColumn("expense_type", F.lower(F.col("expense_type")))\
    .withColumn("ingestion_time", F.current_timestamp())

good_exp.write.format("delta").mode("append").saveAsTable("sales_project.slv.expenses")

bad_exp = exp_parsed.filter((F.col("expense_amount")<0))

bad_exp.write.format("delta").mode("append").saveAsTable("sales_project.brz.bad_records_expenses")


In [0]:
%sql
select * from sales_project.slv.expenses;

In [0]:
%sql
select count(*) from sales_project.slv.expenses;

In [0]:
batch_ingestion = good_exp.agg(F.max(F.col("ingestion_time")).alias("last_ingested")).withColumn("topic", F.lit("expenses"))

batch_ingestion.createOrReplaceTempView("expenses_state")

spark.sql("""
          merge into sales_project.slv.pipeline_metadata t
          using expenses_state s
          on s.topic = t.topic_name
          when matched and (s.last_ingested is not null) and 
          (t.last_ingestion_time is null or s.last_ingested>t.last_ingestion_time)
          then update set t.last_ingestion_time=s.last_ingested
          when not matched then insert(topic_name, last_ingestion_time)
          values(s.topic, s.last_ingested)""")

display(spark.sql("""select * from sales_project.slv.pipeline_metadata"""))

In [0]:
%sql
select * from sales_project.slv.expenses;

In [0]:
%sql
select * from sales_project.brz.regions;

In [0]:
slv_regions_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("region_name", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])


reg_raw = spark.read.table("sales_project.brz.regions").select("value")

reg_parsed = reg_raw.withColumn("value", F.from_json(F.col("value"), slv_regions_schema))

reg_parsed = reg_parsed.select("value.region_id", F.col("value.region_name").alias("city"), "value.ingestion_time")

display(reg_parsed)

In [0]:
reg_parsed.write.format("delta").mode("overwrite").saveAsTable("sales_project.slv.regions")

display(spark.sql("""select * from sales_project.slv.regions;"""))

## Gold

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("sales_project.slv.sales").select("region_id", "product_id", "quantity", "amount", "event_time",
                                                        F.date_format("event_time", "EEEE").alias("day"),
                                                        ((F.weekofyear("event_time"))-
                                                        (F.weekofyear(
                                                         F.date_sub(
                                                             F.to_date("event_time"), 
                                                             F.dayofmonth("event_time") -1)
                                                         )
                                                         )+1).alias("week_of_month")
                                                        )\
    .filter(F.to_date(F.col("event_time"))>=F.dateadd(F.current_date(), -7))\
    .groupBy("region_id", "week_of_month", "day").agg(F.count("*").alias("total_orders"), F.countDistinct("product_id").alias("unique_prods_sold"),
                                                      F.sum(F.col("quantity")).alias("total_units_sold"), F.sum(F.col("amount")).alias("net_revenue"))\
    .withColumn("AOV", F.round(F.col("net_revenue")/F.col("total_orders"),2))

df.write.format("delta").mode("overwrite").saveAsTable("sales_project.gld.daily_sales_summary")


